# 04 Evaluation and Error Analysis

This notebook reviews the saved event-aware test metrics, the optimistic stratified baseline, and the main error analysis conclusion.

The authoritative evaluation code is in `src/evaluate.py` and `src/stratified_baseline.py`. Run commands from the project root. This notebook reads generated outputs and does not rerun evaluation logic.

## Main Event-Aware Test Result

The main realistic evaluation is the event-aware held-out test on failure event 4.

The selected Logistic Regression baseline at threshold `0.61` failed to generalize to held-out event 4:

- `TP = 0`
- `FN = 330`
- Recall = `0.0`
- F1 = `0.0`
- F2 = `0.0`

This is the central evaluation finding. The current model is not deployment-ready.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the project root or notebooks/."

## Event-Aware Test Metrics

The table below loads `results/test_metrics.csv`.

In [ ]:
test_metrics = pd.read_csv(PROJECT_ROOT / "results/test_metrics.csv")
display(test_metrics)

## Stratified Baseline Metrics

The table below loads `results/stratified_baseline_metrics.csv`. This baseline is an optimistic window-level comparison only.

In [ ]:
stratified_baseline_metrics = pd.read_csv(PROJECT_ROOT / "results/stratified_baseline_metrics.csv")
display(stratified_baseline_metrics)

## Evaluation Plots

### Event-aware confusion matrix

![Event-aware confusion matrix](../results/plots/confusion_matrix.png)

### Precision-recall curve

![Precision-recall curve](../results/plots/precision_recall_curve.png)

### Validation probability distribution

![Validation probability distribution](../results/plots/validation_probability_distribution.png)

### Test probability distribution

![Test probability distribution](../results/plots/test_probability_distribution.png)

### Stratified baseline confusion matrix

![Stratified baseline confusion matrix](../results/plots/stratified_baseline_confusion_matrix.png)

In [ ]:
plot_paths = [
    "results/plots/confusion_matrix.png",
    "results/plots/precision_recall_curve.png",
    "results/plots/validation_probability_distribution.png",
    "results/plots/test_probability_distribution.png",
    "results/plots/stratified_baseline_confusion_matrix.png",
]

for relative_path in plot_paths:
    display(Markdown(f"### {relative_path}"))
    display(Image(filename=str(PROJECT_ROOT / relative_path)))

## Why Accuracy Is Misleading

The event-aware test accuracy is high because the event-4 test block is dominated by normal windows. However, the model predicted all 330 true failure-risk windows as normal.

For predictive maintenance, this is a failed result because the most important class was missed. Recall, F2-score, F1-score, and the confusion matrix show the failure clearly, while accuracy hides it.

## Why the Stratified Baseline Looks Strong

The stratified baseline uses a random window-level split. That can mix windows from the same documented failure events across training and test sets.

Those results are useful as an optimistic feature-separability check, but they do not show that the model can generalize to a future independent failure event. The event-aware test remains the main realistic evaluation.